# Joint TNM: neural architecture comparison

This notebook documents an exploratory research iteration using questionnaire scores, V/P/K-coded features and tumour-site indicators. It is part of the [experiment series](../docs/experiments.md).

**Recorded outputs:** the aggregate results below were saved in the original research notebook and retained after privacy review. They have not been recomputed during presentation cleanup. Row previews, per-record target arrays, raw-data exports, local warnings and repetitive epoch logs were removed. Execution counts were cleared because the original execution order was incomplete.

**Re-running:** launch Jupyter from the repository root, install the research dependencies, and supply an authorized final masterchart CSV. Run cells in order. Do not commit newly generated outputs before reviewing them. See [data setup](../data/README.md).

**Interpretation:** these are within-cohort experiments, not a validated clinical model or proof of a semantic relationship. Numeric conversion fills unavailable feature values with zero. Historical pain scaling and site encoding use the complete filtered cohort; fold-local preprocessing and independent validation remain follow-up work.

Rare joint-TNM classes were duplicated **before** the hold-out split in the principal saved runs. This can place duplicated records on both sides of evaluation. These results document method development and should not be used as a performance headline.


## Environment and imports


In [ ]:
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



## Portable local input


In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = Path(os.environ.get("MEDTECH_FINAL_DATA_PATH", REPO_ROOT / "data" / "medtech_final.csv"))
if not DATA_PATH.is_file():
    raise FileNotFoundError("Supply the authorized final masterchart CSV at data/medtech_final.csv or set MEDTECH_FINAL_DATA_PATH.")
df = pd.read_csv(DATA_PATH)


## Cohort dimensions


In [ ]:
df.shape

(226, 329)


## Schema overview


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Columns: 329 entries, S.no to K_Q67
dtypes: float64(54), int64(273), object(2)
memory usage: 581.0+ KB


## Missing-value inspection


In [ ]:
print(df.isnull().sum().sort_values(ascending=False).head(30))


P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


## Select final questionnaire columns


In [ ]:
# Get all column names
cols = df.columns

# Columns to keep
keep_cols = []

for col in cols:
    # If it's a "_final" column → always keep
    if col.endswith('_final'):
        keep_cols.append(col)
    else:
        # Check if corresponding _final exists
        base = col.split('.')[0]  # handles K_Q65.1
        final_col = base + '_final'
        
        if final_col not in cols:
            keep_cols.append(col)

# Keep only selected columns
df = df[keep_cols]

In [ ]:
df.shape

(226, 251)


In [ ]:
print(df.columns.tolist())

['S.no', 'Age', 'weight', 'height', 'Pulse rate', 'Pain', 'Site', 'staging (TNM)', 'Vata', 'Pitta', 'Kapha', 'SF 36', 'Value_Q1', 'Value_Q2', 'Value_Q3', 'Value_Q4', 'Value_Q5', 'Value_Q6', 'Value_Q7', 'Value_Q8', 'Value_Q9', 'Value_Q10', 'Value_Q11', 'Value_Q12', 'Value_Q13', 'Value_Q14', 'Value_Q15', 'Value_Q16', 'Value_Q17', 'Value_Q18', 'Value_Q19', 'Value_Q20', 'Value_Q21', 'Value_Q22', 'Value_Q23', 'Value_Q24', 'Value_Q25', 'Value_Q26', 'Value_Q27', 'Value_Q28', 'Value_Q29', 'Value_Q30', 'Value_Q31', 'Value_Q32', 'Value_Q33', 'Value_Q34', 'Value_Q35', 'Value_Q36', 'V_Q1', 'V_Q2_final', 'V_Q3', 'V_Q4', 'V_Q5', 'V_Q6', 'V_Q7', 'V_Q8', 'V_Q9', 'V_Q10', 'V_Q11', 'V_Q12', 'V_Q13', 'V_Q14', 'V_Q15', 'V_Q16', 'V_Q17_final', 'V_Q18', 'V_Q19', 'V_Q20', 'V_Q21_final', 'V_Q22', 'V_Q23', 'V_Q24', 'V_Q25', 'V_Q26', 'V_Q27', 'V_Q28', 'V_Q29', 'V_Q30', 'V_Q31', 'V_Q32', 'V_Q33', 'V_Q34', 'V_Q35', 'V_Q36', 'V_Q37', 'V_Q38', 'V_Q39', 'V_Q40', 'V_Q41', 'V_Q42', 'V_Q43', 'V_Q44', 'V_Q45', 'V_Q46', 

In [ ]:
for col in df.columns:
    print(f'{col}: {df[col].dtypes}')

S.no: int64
Age: int64
weight: int64
height: int64
Pulse rate: int64
Pain: int64
Site: object
staging (TNM): object
Vata: float64
Pitta: float64
Kapha: float64
SF 36: int64
Value_Q1: int64
Value_Q2: int64
Value_Q3: int64
Value_Q4: int64
Value_Q5: int64
Value_Q6: int64
Value_Q7: int64
Value_Q8: int64
Value_Q9: int64
Value_Q10: int64
Value_Q11: int64
Value_Q12: int64
Value_Q13: int64
Value_Q14: int64
Value_Q15: int64
Value_Q16: int64
Value_Q17: int64
Value_Q18: int64
Value_Q19: int64
Value_Q20: int64
Value_Q21: float64
Value_Q22: int64
Value_Q23: float64
Value_Q24: float64
Value_Q25: float64
Value_Q26: float64
Value_Q27: float64
Value_Q28: float64
Value_Q29: float64
Value_Q30: float64
Value_Q31: float64
Value_Q32: int64
Value_Q33: int64
Value_Q34: int64
Value_Q35: int64
Value_Q36: int64
V_Q1: int64
V_Q2_final: int64
V_Q3: int64
V_Q4: float64
V_Q5: int64
V_Q6: int64
V_Q7: int64
V_Q8: int64
V_Q9: float64
V_Q10: int64
V_Q11: int64
V_Q12: int64
V_Q13: int64
V_Q14: int64
V_Q15: int64
V_Q16: i

In [ ]:
df['Site'].value_counts()

Site
Oral cavity                 62
Supraglottic                39
Glottic                     33
Hypopharyngeal              30
Oropharyngeal               23
Sinonasal carcinoma         10
Nasopharyngeal Carcinoma     8
Maxillary sinus              5
Sinonasal Carcinoma          4
Glottis                      3
Esophagus                    2
Esophageal                   1
Sinonasal chondrosarcoma     1
Oropharynx                   1
sinonasal carcinoma          1
Sinonasal Sarcoma            1
Nasopharyngeal carcinoma     1
Transglottic                 1
Name: count, dtype: int64


## Normalize joint TNM labels


In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# -----------------------------
# TNM normalization
# -----------------------------
def normalize_stage_tnm(raw):
    if pd.isna(raw):
        return np.nan

    s = str(raw).upper()

    # Remove unwanted characters (keep only TNM-related)
    s = re.sub(r'[^TNM0-9ABCIS]', '', s)

    # Extract T, N, M
    t_match = re.search(r'T(IS|[0-4][ABC]?)', s)
    n_match = re.search(r'N([0-3])[ABC]?', s)
    m_match = re.search(r'M([01])', s)

    if not (t_match and n_match and m_match):
        return np.nan

    t_token = t_match.group(1).lower()
    n_token = n_match.group(1)
    m_token = m_match.group(1)

    return f"T{t_token}N{n_token}M{m_token}"


# -----------------------------
# Apply on dataset
# -----------------------------

df["stage_clean"] = df["staging (TNM)"].apply(normalize_stage_tnm)

# Keep only valid rows
df = df[df["stage_clean"].notna()].copy()

# -----------------------------
# Active classes in dataset
# -----------------------------
active_stages = sorted(df["stage_clean"].unique())

print("Active TNM classes in dataset:", len(active_stages))
print(active_stages[:10])

# -----------------------------
# Label Encoding
# -----------------------------
stage_le = LabelEncoder()
y_stage = stage_le.fit_transform(df["stage_clean"])

print("\nEncoded classes:", len(stage_le.classes_))
print(stage_le.classes_[:10])

Active TNM classes in dataset: 21
['T1N1M0', 'T1aN0M0', 'T1bN0M0', 'T2N0M0', 'T2N1M0', 'T2N2M0', 'T2N3M0', 'T3N0M0', 'T3N1M0', 'T3N2M0']

Encoded classes: 21
['T1N1M0' 'T1aN0M0' 'T1bN0M0' 'T2N0M0' 'T2N1M0' 'T2N2M0' 'T2N3M0' 'T3N0M0'
 'T3N1M0' 'T3N2M0']


## Historical numeric filtering and scaling

This preserves the historical complete-cohort min-max scaling, including Pain. For confirmatory evaluation, fit preprocessing on training folds and specify the outcome scale independently.


In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


# -----------------------------
# 3. Filter valid rows
# -----------------------------
numeric_cols = ["Pain", "weight", "height", "Pulse rate"]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

mask = df["stage_clean"].notna()
for col in numeric_cols:
    mask &= df[col].notna()

df = df.loc[mask].copy()

# -----------------------------
# 4. Encode TNM (replace original column)
# -----------------------------
stage_le = LabelEncoder()
df["staging (TNM)"] = stage_le.fit_transform(df["stage_clean"])

print("Number of stage classes:", len(stage_le.classes_))
print("Sample classes:", stage_le.classes_[:10])

# -----------------------------
# 5. Min-Max scaling (replace original columns)
# -----------------------------
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# -----------------------------
# 6. Cleanup (remove helper column)
# -----------------------------
df.drop(columns=["stage_clean"], inplace=True)

# -----------------------------
# Final dataset ready
# -----------------------------
print("\nFinal shape:", df.shape)


Final shape: (226, 251)


## Encode tumour-site categories


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# ensure df is DataFrame
print(type(df))

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[["Site"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Site"]), index=df.index
)

df = pd.concat([df, encoded_df], axis=1)
df.drop("Site", axis=1, inplace=True)

## Build VALUE, VPK and ALL feature settings


In [ ]:
import pandas as pd
import re

def prepare_dataset(df, mode="value"):
    """
    mode options:
    - "value"     → Value_Q1 to Value_Q36
    - "vpk"       → V_Q*, P_Q*, K_Q*
    - "all"       → Value_Q* + V_Q* + P_Q* + K_Q* + Site_*

    Returns:
      X      : feature matrix
      y_stg  : staging (TNM)
      y_pain : Pain
    """

    df = df.copy()

    # --- 1. Clean column names ---
    df.columns = df.columns.str.strip()
    df.columns = [re.sub(r'\.\d+$', '', col) for col in df.columns]
    df.columns = [col.replace('-', '_') for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    # --- 2. Targets ---
    if "staging (TNM)" not in df.columns:
        raise ValueError("Target column 'staging (TNM)' not found")
    if "Pain" not in df.columns:
        raise ValueError("Target column 'Pain' not found")

    y_stg = pd.to_numeric(df["staging (TNM)"], errors="coerce")
    y_pain = pd.to_numeric(df["Pain"], errors="coerce")

    # keep only rows where both targets are valid
    mask = y_stg.notna() & y_pain.notna()
    df = df.loc[mask].copy()
    y_stg = y_stg.loc[mask].astype(int)
    y_stg = y_stg.astype(int)
    y_pain = y_pain.loc[mask].astype(float)

    # --- 3. Select feature columns based on mode ---
    if mode == "value":
        feature_cols = [col for col in df.columns if col.startswith("Value_Q")]

    elif mode == "vpk":
        feature_cols = [
            col for col in df.columns
            if col.startswith("V_Q") or col.startswith("P_Q") or col.startswith("K_Q")
        ]

    elif mode == "all":
        feature_cols = [
            col for col in df.columns
            if col.startswith("Value_Q")
            or col.startswith("V_Q")
            or col.startswith("P_Q")
            or col.startswith("K_Q")
        ]

    else:
        raise ValueError("Invalid mode. Choose from: value, vpk, all")

    # --- 4. Add existing Site_ one-hot columns ---
    site_cols = [col for col in df.columns if col.startswith("Site_")]

    # --- 5. Final feature matrix ---
    final_cols = feature_cols + site_cols
    X = (
        df[final_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return (
        np.nan_to_num(np.asarray(X, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0),
        np.asarray(y_stg),
        np.asarray(y_pain),
    )


# Example usage
X1, y_stg1, y_pain1 = prepare_dataset(df, mode="value")
X2, y_stg2, y_pain2 = prepare_dataset(df, mode="vpk")
X3, y_stg3, y_pain3 = prepare_dataset(df, mode="all")

print(X1.shape, X2.shape, X3.shape)
print(len(np.unique(y_stg1)), y_pain1.min(), y_pain1.max())

(226, 54) (226, 220) (226, 256)
21 0.0 1.0


## Rare-class handling


In [ ]:
import numpy as np
from collections import Counter

def fix_single_sample_classes(X, y_stg, y_pain, min_count=3):
    """
    Ensures each class in y_stg has at least `min_count` samples
    by duplicating rows.

    Parameters:
        X       : feature matrix (numpy array)
        y_stg   : class labels (numpy array)
        y_pain  : regression target (numpy array)
        min_count : minimum samples required per class

    Returns:
        X_new, y_stg_new, y_pain_new
    """

    X = np.asarray(X)
    y_stg = np.asarray(y_stg)
    y_pain = np.asarray(y_pain)

    class_counts = Counter(y_stg)

    X_list = [X]
    ystg_list = [y_stg]
    ypain_list = [y_pain]

    for cls, count in class_counts.items():
        if count < min_count:
            needed = min_count - count

            # indices of this class
            idx = np.where(y_stg == cls)[0]

            # randomly duplicate existing rows
            dup_idx = np.random.choice(idx, size=needed, replace=True)

            X_list.append(X[dup_idx])
            ystg_list.append(y_stg[dup_idx])
            ypain_list.append(y_pain[dup_idx])

    # combine all
    X_new = np.vstack(X_list)
    y_stg_new = np.concatenate(ystg_list)
    y_pain_new = np.concatenate(ypain_list)

    return X_new, y_stg_new, y_pain_new

## Multitask architecture definitions


In [ ]:
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class MultiTaskNet(nn.Module):
    def __init__(self, input_dim, n_classes, hidden_dims=(64, 32, 16), dropout=0.05):
        super().__init__()

        h1, h2, h3 = hidden_dims

        self.shared = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.ReLU()
        )

        self.stage_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, n_classes)
        )

        self.pain_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, 1)
        )

    def forward(self, x):
        shared = self.shared(x)
        stage_logits = self.stage_head(shared)
        pain_pred = self.pain_head(shared)
        return stage_logits, pain_pred


def build_model(input_dim, n_classes, arch="big"):
    """
    arch = "big"   -> 64-32-16
    arch = "small" -> 32-16-8
    """
    if arch == "big":
        hidden_dims = (64, 32, 16)
        dropout = 0.05
    elif arch == "small":
        hidden_dims = (32, 16, 8)
        dropout = 0.03
    else:
        raise ValueError("arch must be 'big' or 'small'")

    model = MultiTaskNet(
        input_dim=input_dim,
        n_classes=n_classes,
        hidden_dims=hidden_dims,
        dropout=dropout
    )
    return model.to(device)

Using device: cpu


## Aggregate diagnostic metrics


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    confusion_matrix
)

def regression_debug_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    pred_var = float(np.var(y_pred))
    true_var = float(np.var(y_true))
    resid = y_true - y_pred
    resid_std = float(np.std(resid))

    return {
        "MAE": float(mae),
        "MSE": float(mse),
        "RMSE": rmse,
        "R2": float(r2),
        "ExplainedVariance": float(evs),
        "PredVariance": pred_var,
        "TrueVariance": true_var,
        "ResidualStd": resid_std,
    }


def classification_debug_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    out = {
        "Accuracy": float(acc),
        "BalancedAccuracy": float(bal_acc),
        "MacroF1": float(macro_f1),
        "ConfusionMatrix": confusion_matrix(y_true, y_pred)
    }

    if y_prob is not None:
        y_prob = np.asarray(y_prob)
        max_conf = np.max(y_prob, axis=1)
        entropy = -np.sum(y_prob * np.log(np.clip(y_prob, 1e-12, 1.0)), axis=1)
        out["MeanSoftmaxConfidence"] = float(np.mean(max_conf))
        out["MedianSoftmaxConfidence"] = float(np.median(max_conf))
        out["MeanEntropy"] = float(np.mean(entropy))

    return out


def print_metrics_block(title, cls_metrics, reg_metrics):
    print(f"\n===== {title} =====")
    print(
        f"Stage | Acc: {cls_metrics['Accuracy']:.4f} | "
        f"BalAcc: {cls_metrics['BalancedAccuracy']:.4f} | "
        f"MacroF1: {cls_metrics['MacroF1']:.4f}"
    )
    if "MeanSoftmaxConfidence" in cls_metrics:
        print(
            f"Stage Confidence | Mean: {cls_metrics['MeanSoftmaxConfidence']:.4f} | "
            f"Median: {cls_metrics['MedianSoftmaxConfidence']:.4f} | "
            f"Entropy: {cls_metrics['MeanEntropy']:.4f}"
        )

    print(
        f"Pain  | MAE: {reg_metrics['MAE']:.4f} | "
        f"RMSE: {reg_metrics['RMSE']:.4f} | "
        f"MSE: {reg_metrics['MSE']:.4f} | "
        f"R2: {reg_metrics['R2']:.4f} | "
        f"EVS: {reg_metrics['ExplainedVariance']:.4f}"
    )
    print(
        f"Pain Variance | Pred: {reg_metrics['PredVariance']:.4f} | "
        f"True: {reg_metrics['TrueVariance']:.4f} | "
        f"ResidualStd: {reg_metrics['ResidualStd']:.4f}"
    )
    print("Confusion matrix:\n", cls_metrics["ConfusionMatrix"])

## Training and evaluation runner


In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

def run_model(
    X,
    y_stg,
    y_pain,
    dataset_name="dataset",
    arch="big",
    min_count=3,
    test_size=0.20,
    random_state=42,
    batch_size=16,
    epochs=120,
    lr=3e-3,
    weight_decay=1e-5,
    stage_loss_w=1.0,
    pain_loss_w=0.7
):
    """
    Trains on train set only, evaluates on test set.
    Uses:
      - fix_single_sample_classes before splitting
      - AdamW
      - OneCycleLR with cosine annealing
      - no validation split
    """

    # ---------------------------------------------------
    # 1) Sanity checks + replication for rare classes
    # ---------------------------------------------------
    X = np.asarray(X, dtype=np.float32)
    y_stg = np.asarray(y_stg, dtype=np.int64).reshape(-1)
    y_pain = np.asarray(y_pain, dtype=np.float32).reshape(-1, 1)

    print(f"\n--- {dataset_name.upper()} ---")
    print("Original shape:", X.shape)
    print("Original class counts:", dict(Counter(y_stg)))

    X_rep, y_stg_rep, y_pain_rep = fix_single_sample_classes(
        X, y_stg, y_pain, min_count=min_count
    )

    print("After replication shape:", X_rep.shape)
    print("After replication class counts:", dict(Counter(y_stg_rep)))

    # ---------------------------------------------------
    # 2) Train/test split
    # ---------------------------------------------------
    X_train, X_test, y_train_stg, y_test_stg, y_train_pain, y_test_pain = train_test_split(
        X_rep,
        y_stg_rep,
        y_pain_rep,
        test_size=test_size,
        random_state=random_state,
        stratify=y_stg_rep
    )

    print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
    print("Train class counts:", dict(Counter(y_train_stg)))
    print("Test class counts:", dict(Counter(y_test_stg)))

    # ---------------------------------------------------
    # 3) DataLoaders
    # ---------------------------------------------------
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_stg_t = torch.tensor(y_train_stg, dtype=torch.long)
    y_train_pain_t = torch.tensor(y_train_pain, dtype=torch.float32)

    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_stg_t = torch.tensor(y_test_stg, dtype=torch.long)
    y_test_pain_t = torch.tensor(y_test_pain, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_stg_t, y_train_pain_t)
    test_ds = TensorDataset(X_test_t, y_test_stg_t, y_test_pain_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # ---------------------------------------------------
    # 4) Model + losses + optimizer + scheduler
    # ---------------------------------------------------
    n_classes = len(np.unique(y_stg_rep))
    model = build_model(input_dim=X.shape[1], n_classes=n_classes, arch=arch)

    stage_criterion = nn.CrossEntropyLoss()
    pain_criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = max(1, epochs * len(train_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=0.25,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0
    )

    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ---------------------------------------------------
    # 5) Training loop
    # ---------------------------------------------------
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        stage_correct = 0
        stage_total = 0
        pain_sq_err_sum = 0.0
        n_seen = 0

        for xb, ysb, ypb in train_loader:
            xb = xb.to(device)
            ysb = ysb.to(device)
            ypb = ypb.to(device).squeeze(-1)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                stage_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_stage = stage_criterion(stage_logits, ysb)
                loss_pain = pain_criterion(pain_pred, ypb)
                loss = stage_loss_w * loss_stage + pain_loss_w * loss_pain

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * xb.size(0)

            pred_stage = torch.argmax(stage_logits, dim=1)
            stage_correct += (pred_stage == ysb).sum().item()
            stage_total += ysb.size(0)

            pain_sq_err_sum += torch.sum((pain_pred.detach() - ypb) ** 2).item()
            n_seen += ysb.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = stage_correct / max(1, stage_total)
        train_rmse = np.sqrt(pain_sq_err_sum / max(1, n_seen))

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"lr={current_lr:.6f} | "
            f"train_loss={train_loss:.4f} | "
            f"train_stage_acc={train_acc:.4f} | "
            f"train_pain_rmse={train_rmse:.4f}"
        )

    # ---------------------------------------------------
    # 6) Test evaluation
    # ---------------------------------------------------
    model.eval()
    all_stage_true, all_stage_pred, all_stage_prob = [], [], []
    all_pain_true, all_pain_pred = [], []

    with torch.no_grad():
        for xb, ysb, ypb in test_loader:
            xb = xb.to(device)
            ysb = ysb.to(device)
            ypb = ypb.to(device).squeeze(-1)

            stage_logits, pain_pred = model(xb)
            stage_prob = torch.softmax(stage_logits, dim=1)

            stage_cls = torch.argmax(stage_prob, dim=1)

            all_stage_true.extend(ysb.cpu().numpy().tolist())
            all_stage_pred.extend(stage_cls.cpu().numpy().tolist())
            all_stage_prob.extend(stage_prob.cpu().numpy().tolist())

            all_pain_true.extend(ypb.cpu().numpy().tolist())
            all_pain_pred.extend(pain_pred.squeeze(-1).cpu().numpy().tolist())

    cls_metrics = classification_debug_metrics(
        np.array(all_stage_true),
        np.array(all_stage_pred),
        np.array(all_stage_prob)
    )
    reg_metrics = regression_debug_metrics(
        np.array(all_pain_true),
        np.array(all_pain_pred)
    )

    print_metrics_block(dataset_name, cls_metrics, reg_metrics)

    return {
        "model": model,
        "classification": cls_metrics,
        "regression": reg_metrics
    }

## Recorded architecture run: VALUE / big


In [ ]:
results_value_big = run_model(X1, y_stg1, y_pain1, dataset_name="value", arch="big")
# results_vpk_big   = run_model(X2, y_stg2, y_pain2, dataset_name="vpk", arch="big")
# results_all_big   = run_model(X3, y_stg3, y_pain3, dataset_name="all", arch="big")

# results_value_small = run_model(X1, y_stg1, y_pain1, dataset_name="value_small", arch="small")
# results_vpk_small   = run_model(X2, y_stg2, y_pain2, dataset_name="vpk_small", arch="small")
# results_all_small   = run_model(X3, y_stg3, y_pain3, dataset_name="all_small", arch="small")


--- VALUE ---
Original shape: (226, 54)
Original class counts: {5: 13, 8: 90, 3: 6, 20: 1, 4: 9, 2: 3, 9: 7, 11: 1, 7: 32, 18: 26, 16: 14, 19: 2, 6: 7, 1: 1, 12: 2, 13: 1, 0: 1, 10: 3, 14: 1, 17: 5, 15: 1}
After replication shape: (242, 54)
After replication class counts: {5: 13, 8: 90, 3: 6, 20: 3, 4: 9, 2: 3, 9: 7, 11: 3, 7: 32, 18: 26, 16: 14, 19: 3, 6: 7, 1: 3, 12: 3, 13: 3, 0: 3, 10: 3, 14: 3, 17: 5, 15: 3}
Train shape: (193, 54) | Test shape: (49, 54)
Train class counts: {9: 6, 8: 72, 18: 21, 7: 26, 16: 11, 6: 6, 20: 3, 3: 5, 19: 3, 12: 2, 14: 2, 1: 2, 11: 2, 17: 4, 4: 7, 0: 3, 5: 10, 2: 2, 10: 2, 13: 2, 15: 2}
Test class counts: {7: 6, 5: 3, 18: 5, 16: 3, 2: 1, 8: 18, 14: 1, 4: 2, 10: 1, 15: 1, 11: 1, 13: 1, 3: 1, 9: 1, 17: 1, 1: 1, 6: 1, 12: 1}



===== value =====
Stage | Acc: 0.3878 | BalAcc: 0.2068 | MacroF1: 0.1799
Stage Confidence | Mean: 0.5166 | Median: 0.4445 | Entropy: 1.4413
Pain  | MAE: 0.1397 | RMSE: 0.1723 | MSE: 0.0297 | R2: 0.5071 | EVS: 0.5301
Pain Variance | Pred: 0.0320 | True: 0.0602 | ResidualStd: 0.1682
Confusion matrix:
 [[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  1  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  2  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  0  0  0  0  3  3  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  4 13  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  1 

## Recorded architecture run: VPK / big


In [ ]:
results_vpk_big   = run_model(X2, y_stg2, y_pain2, dataset_name="vpk", arch="big")


--- VPK ---
Original shape: (226, 220)
Original class counts: {5: 13, 8: 90, 3: 6, 20: 1, 4: 9, 2: 3, 9: 7, 11: 1, 7: 32, 18: 26, 16: 14, 19: 2, 6: 7, 1: 1, 12: 2, 13: 1, 0: 1, 10: 3, 14: 1, 17: 5, 15: 1}
After replication shape: (242, 220)
After replication class counts: {5: 13, 8: 90, 3: 6, 20: 3, 4: 9, 2: 3, 9: 7, 11: 3, 7: 32, 18: 26, 16: 14, 19: 3, 6: 7, 1: 3, 12: 3, 13: 3, 0: 3, 10: 3, 14: 3, 17: 5, 15: 3}
Train shape: (193, 220) | Test shape: (49, 220)
Train class counts: {9: 6, 8: 72, 18: 21, 7: 26, 16: 11, 6: 6, 20: 3, 3: 5, 19: 3, 12: 2, 14: 2, 1: 2, 11: 2, 17: 4, 4: 7, 0: 3, 5: 10, 2: 2, 10: 2, 13: 2, 15: 2}
Test class counts: {7: 6, 5: 3, 18: 5, 16: 3, 2: 1, 8: 18, 14: 1, 4: 2, 10: 1, 15: 1, 11: 1, 13: 1, 3: 1, 9: 1, 17: 1, 1: 1, 6: 1, 12: 1}



===== vpk =====
Stage | Acc: 0.3878 | BalAcc: 0.3938 | MacroF1: 0.3357
Stage Confidence | Mean: 0.5978 | Median: 0.5689 | Entropy: 1.1397
Pain  | MAE: 0.1713 | RMSE: 0.2190 | MSE: 0.0479 | R2: 0.2036 | EVS: 0.2044
Pain Variance | Pred: 0.0196 | True: 0.0602 | ResidualStd: 0.2188
Confusion matrix:
 [[ 1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  1  0  0  0  0  1  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  2  2  0  0  0  0  0  0  0  2  0  0]
 [ 0  0  0  2  0  0  2 10  1  0  0  0  0  0  0  0  0  3]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0

## Recorded architecture run: ALL / big


In [ ]:
results_all_big   = run_model(X3, y_stg3, y_pain3, dataset_name="all", arch="big")


--- ALL ---
Original shape: (226, 256)
Original class counts: {5: 13, 8: 90, 3: 6, 20: 1, 4: 9, 2: 3, 9: 7, 11: 1, 7: 32, 18: 26, 16: 14, 19: 2, 6: 7, 1: 1, 12: 2, 13: 1, 0: 1, 10: 3, 14: 1, 17: 5, 15: 1}
After replication shape: (242, 256)
After replication class counts: {5: 13, 8: 90, 3: 6, 20: 3, 4: 9, 2: 3, 9: 7, 11: 3, 7: 32, 18: 26, 16: 14, 19: 3, 6: 7, 1: 3, 12: 3, 13: 3, 0: 3, 10: 3, 14: 3, 17: 5, 15: 3}
Train shape: (193, 256) | Test shape: (49, 256)
Train class counts: {9: 6, 8: 72, 18: 21, 7: 26, 16: 11, 6: 6, 20: 3, 3: 5, 19: 3, 12: 2, 14: 2, 1: 2, 11: 2, 17: 4, 4: 7, 0: 3, 5: 10, 2: 2, 10: 2, 13: 2, 15: 2}
Test class counts: {7: 6, 5: 3, 18: 5, 16: 3, 2: 1, 8: 18, 14: 1, 4: 2, 10: 1, 15: 1, 11: 1, 13: 1, 3: 1, 9: 1, 17: 1, 1: 1, 6: 1, 12: 1}



===== all =====
Stage | Acc: 0.3061 | BalAcc: 0.3815 | MacroF1: 0.3210
Stage Confidence | Mean: 0.6788 | Median: 0.6612 | Entropy: 0.8463
Pain  | MAE: 0.1380 | RMSE: 0.1716 | MSE: 0.0294 | R2: 0.5108 | EVS: 0.5260
Pain Variance | Pred: 0.0404 | True: 0.0602 | ResidualStd: 0.1689
Confusion matrix:
 [[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 2 0 2 0 0]
 [0 0 1 1 0 2 2 6 2 0 0 0 0 0 0 1 1 1 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 

## Recorded architecture run: VALUE / small


In [ ]:
results_value_small = run_model(X1, y_stg1, y_pain1, dataset_name="value_small", arch="small")



--- VALUE_SMALL ---
Original shape: (226, 54)
Original class counts: {5: 13, 8: 90, 3: 6, 20: 1, 4: 9, 2: 3, 9: 7, 11: 1, 7: 32, 18: 26, 16: 14, 19: 2, 6: 7, 1: 1, 12: 2, 13: 1, 0: 1, 10: 3, 14: 1, 17: 5, 15: 1}
After replication shape: (242, 54)
After replication class counts: {5: 13, 8: 90, 3: 6, 20: 3, 4: 9, 2: 3, 9: 7, 11: 3, 7: 32, 18: 26, 16: 14, 19: 3, 6: 7, 1: 3, 12: 3, 13: 3, 0: 3, 10: 3, 14: 3, 17: 5, 15: 3}
Train shape: (193, 54) | Test shape: (49, 54)
Train class counts: {9: 6, 8: 72, 18: 21, 7: 26, 16: 11, 6: 6, 20: 3, 3: 5, 19: 3, 12: 2, 14: 2, 1: 2, 11: 2, 17: 4, 4: 7, 0: 3, 5: 10, 2: 2, 10: 2, 13: 2, 15: 2}
Test class counts: {7: 6, 5: 3, 18: 5, 16: 3, 2: 1, 8: 18, 14: 1, 4: 2, 10: 1, 15: 1, 11: 1, 13: 1, 3: 1, 9: 1, 17: 1, 1: 1, 6: 1, 12: 1}



===== value_small =====
Stage | Acc: 0.3673 | BalAcc: 0.0679 | MacroF1: 0.0463
Stage Confidence | Mean: 0.4088 | Median: 0.3717 | Entropy: 1.7889
Pain  | MAE: 0.1234 | RMSE: 0.1512 | MSE: 0.0229 | R2: 0.6200 | EVS: 0.6342
Pain Variance | Pred: 0.0344 | True: 0.0602 | ResidualStd: 0.1484
Confusion matrix:
 [[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  1  0]
 [ 0  0  0  0  0  0  1  2  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  2  4  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1 16  0  0  0  0  0  0  0  1  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1]

## Recorded architecture run: VPK / small


In [ ]:
results_vpk_small   = run_model(X2, y_stg2, y_pain2, dataset_name="vpk_small", arch="small")


--- VPK_SMALL ---
Original shape: (226, 220)
Original class counts: {5: 13, 8: 90, 3: 6, 20: 1, 4: 9, 2: 3, 9: 7, 11: 1, 7: 32, 18: 26, 16: 14, 19: 2, 6: 7, 1: 1, 12: 2, 13: 1, 0: 1, 10: 3, 14: 1, 17: 5, 15: 1}
After replication shape: (242, 220)
After replication class counts: {5: 13, 8: 90, 3: 6, 20: 3, 4: 9, 2: 3, 9: 7, 11: 3, 7: 32, 18: 26, 16: 14, 19: 3, 6: 7, 1: 3, 12: 3, 13: 3, 0: 3, 10: 3, 14: 3, 17: 5, 15: 3}
Train shape: (193, 220) | Test shape: (49, 220)
Train class counts: {9: 6, 8: 72, 18: 21, 7: 26, 16: 11, 6: 6, 20: 3, 3: 5, 19: 3, 12: 2, 14: 2, 1: 2, 11: 2, 17: 4, 4: 7, 0: 3, 5: 10, 2: 2, 10: 2, 13: 2, 15: 2}
Test class counts: {7: 6, 5: 3, 18: 5, 16: 3, 2: 1, 8: 18, 14: 1, 4: 2, 10: 1, 15: 1, 11: 1, 13: 1, 3: 1, 9: 1, 17: 1, 1: 1, 6: 1, 12: 1}



===== vpk_small =====
Stage | Acc: 0.3265 | BalAcc: 0.1241 | MacroF1: 0.1088
Stage Confidence | Mean: 0.4130 | Median: 0.3745 | Entropy: 1.8203
Pain  | MAE: 0.1793 | RMSE: 0.2281 | MSE: 0.0520 | R2: 0.1354 | EVS: 0.1426
Pain Variance | Pred: 0.0206 | True: 0.0602 | ResidualStd: 0.2272
Confusion matrix:
 [[ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  0  0  0  2  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1  2  0  0  0  0  0  0  0  0  0  3]
 [ 0  0  0  0  0  0  6 12  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0

## Recorded architecture run: ALL / small


In [ ]:
results_all_small   = run_model(X3, y_stg3, y_pain3, dataset_name="all_small", arch="small")


--- ALL_SMALL ---
Original shape: (226, 256)
Original class counts: {5: 13, 8: 90, 3: 6, 20: 1, 4: 9, 2: 3, 9: 7, 11: 1, 7: 32, 18: 26, 16: 14, 19: 2, 6: 7, 1: 1, 12: 2, 13: 1, 0: 1, 10: 3, 14: 1, 17: 5, 15: 1}
After replication shape: (242, 256)
After replication class counts: {5: 13, 8: 90, 3: 6, 20: 3, 4: 9, 2: 3, 9: 7, 11: 3, 7: 32, 18: 26, 16: 14, 19: 3, 6: 7, 1: 3, 12: 3, 13: 3, 0: 3, 10: 3, 14: 3, 17: 5, 15: 3}
Train shape: (193, 256) | Test shape: (49, 256)
Train class counts: {9: 6, 8: 72, 18: 21, 7: 26, 16: 11, 6: 6, 20: 3, 3: 5, 19: 3, 12: 2, 14: 2, 1: 2, 11: 2, 17: 4, 4: 7, 0: 3, 5: 10, 2: 2, 10: 2, 13: 2, 15: 2}
Test class counts: {7: 6, 5: 3, 18: 5, 16: 3, 2: 1, 8: 18, 14: 1, 4: 2, 10: 1, 15: 1, 11: 1, 13: 1, 3: 1, 9: 1, 17: 1, 1: 1, 6: 1, 12: 1}



===== all_small =====
Stage | Acc: 0.2653 | BalAcc: 0.2704 | MacroF1: 0.2338
Stage Confidence | Mean: 0.5805 | Median: 0.5648 | Entropy: 1.2467
Pain  | MAE: 0.1196 | RMSE: 0.1412 | MSE: 0.0199 | R2: 0.6689 | EVS: 0.6720
Pain Variance | Pred: 0.0408 | True: 0.0602 | ResidualStd: 0.1405
Confusion matrix:
 [[0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 2 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 2 4 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 2 0 1 4 6 0 0 0 0 0 0 0 0 0 3 1]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 